In [1]:
import warnings
# Ignore all warnings

# Enable interactive mode with a scrollable plot
%matplotlib qt
import numpy as np
import glob
import matplotlib.pyplot as plt
import sys
import pickle as pkl
sys.path.append("..")
import mne

from scipy.io import loadmat
import socket
import pickle as pkl
import copy
import nibabel as nib
import json
import nibabel as nib


%load_ext autoreload
%autoreload 2

In [2]:
device_name = socket.gethostname()
print(device_name)

DESKTOP-I0FNO1F


In [3]:
if device_name == "LAPTOP-LRI6535I":
    path_analysis_data = "E:/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/"
    path_analysis_data_new = "E:/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"
    plots_path = "E:/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/analysis_plots/"

if device_name == "kritrim":
    path_analysis_data = "C:/Users/manda/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"

if device_name == "DESKTOP-I0FNO1F":
    path_analysis_data = "D:/Taran/OneDrive - Radboud Universiteit/Documents/PDoCDataAnalysis/analysis_data_new2/"
    path_analysis_data_new = "D:/Taran/OneDrive - Radboud Universiteit/Documents/pdoc_data_analysis/analysis_data/"
    # plots_path = "D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots/"
    plots_path = "D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/"

In [4]:
from helpers.source_level_psd_helpers import *
from helpers.frequency_analysis import compute_psd_mne
from helpers.helper_functions import get_matched_legths, split_half_latex_table
from helpers.topo_eeg import get_fdr_p_values, get_band_power_difference_all_subjects_significance_aperiodic_fdr, get_channel_wise_fooof
from helpers.helper_functions import get_fooof_aperiodic, get_fooof
from helpers.group_level_helpers import group_level_bootstrap_test_network, build_channel_style_arrays, latex_group_vs_individual_table

def store_mne_raw(eeg_filtered, path_folder, path_data, ovwrite=False):
    eeg_filtered.save(path_folder + path_data, overwrite=ovwrite)

def load_mne_raw(path_folder, path_data):
    return mne.io.read_raw_fif(path_folder + path_data, preload=True)


Python Executable Path: 
c:\Users\specsdcn\anaconda3\envs\pdoc\python.exe


d:\Taran\OneDrive - Radboud Universiteit\Documents\music_doc_study\analysis\..\helpers\helper_functions.py:9: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  import fooof


In [5]:
# import numpy as np
# import helpers.topo_eeg as _topo
# import helpers.source_level_psd_helpers as _slph
# import helpers.group_level_helpers as _glh

# def _no_multipletests(pvals, alpha=0.05, method=None, **kw):
#     p = np.asarray(pvals, dtype=float)
#     return p < alpha, p, alpha, alpha          # multipletests returns 4 values

# def _no_fdrcorrection(pvals, alpha=0.05, method="indep", **kw):
#     p = np.asarray(pvals, dtype=float)
#     return p < alpha, p                        # fdrcorrection returns 2 values

# for _mod in (_topo, _slph, _glh):
#     if hasattr(_mod, "multipletests"):
#         _mod.multipletests = _no_multipletests
#     if hasattr(_mod, "fdrcorrection"):
#         _mod.fdrcorrection = _no_fdrcorrection

# print("patched:", [(m.__name__, hasattr(m, "multipletests"), hasattr(m, "fdrcorrection"))
#                    for m in (_topo, _slph, _glh)])

In [6]:
import warnings
warnings.filterwarnings('ignore')

bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30),
    "gamma low": (30, 45),
}



pat_state_dict = {
    'sub01': 'UWS',
    'sub03': 'MCS+',
    'sub04': 'MCS-',
    'sub07': 'MCS-',
    'sub09': 'MCS-',
    'sub10': 'MCS-',
    'sub12': 'MCS-',
}



sfreq = 512
paradigms = {0: "noise", 1: "music"}
pat_state = {"sub01": "UWS", "sub03": "MCS+", "sub04": "MCS-", "sub07":"MCS-", "sub09":"MCS-", "sub10":"MCS-", "sub12":"MCS-"}

NETWORKS = ["Sensorimotor Network", "Default Mode Network", "Frontoparietal Network"]
NET_ABBR = {"Sensorimotor Network": "SMN", "Default Mode Network": "DMN", "Frontoparietal Network": "FPN"}


#### Load AAL Atlas

In [7]:
aal_dir = path_analysis_data_new + "eeg_source/"


with open(f'{aal_dir}AAL_labels.json', 'r') as f:
    aal_labels = json.load(f)

In [8]:
aal_labels

['Precentral_L',
 'Precentral_R',
 'Frontal_Sup_L',
 'Frontal_Sup_R',
 'Frontal_Sup_Orb_L',
 'Frontal_Sup_Orb_R',
 'Frontal_Mid_L',
 'Frontal_Mid_R',
 'Frontal_Mid_Orb_L',
 'Frontal_Mid_Orb_R',
 'Frontal_Inf_Oper_L',
 'Frontal_Inf_Oper_R',
 'Frontal_Inf_Tri_L',
 'Frontal_Inf_Tri_R',
 'Frontal_Inf_Orb_L',
 'Frontal_Inf_Orb_R',
 'Rolandic_Oper_L',
 'Rolandic_Oper_R',
 'Supp_Motor_Area_L',
 'Supp_Motor_Area_R',
 'Olfactory_L',
 'Olfactory_R',
 'Frontal_Sup_Medial_L',
 'Frontal_Sup_Medial_R',
 'Frontal_Med_Orb_L',
 'Frontal_Med_Orb_R',
 'Rectus_L',
 'Rectus_R',
 'Insula_L',
 'Insula_R',
 'Cingulum_Ant_L',
 'Cingulum_Ant_R',
 'Cingulum_Mid_L',
 'Cingulum_Mid_R',
 'Cingulum_Post_L',
 'Cingulum_Post_R',
 'Hippocampus_L',
 'Hippocampus_R',
 'ParaHippocampal_L',
 'ParaHippocampal_R',
 'Amygdala_L',
 'Amygdala_R',
 'Calcarine_L',
 'Calcarine_R',
 'Cuneus_L',
 'Cuneus_R',
 'Lingual_L',
 'Lingual_R',
 'Occipital_Sup_L',
 'Occipital_Sup_R',
 'Occipital_Mid_L',
 'Occipital_Mid_R',
 'Occipital_Inf_L'

LOAD FOOOF

In [9]:
fooof_path = path_analysis_data_new + "fooof_all_sources_vector_nofilt/"
fooof_files = glob.glob(fooof_path + "*.pkl")

In [10]:
from pathlib import Path
eeg_fooof_all_subs = {sub: {} for sub in pat_state_dict}

for fl in fooof_files:
    file_stem = Path(fl).stem
    file_name = Path(fl).name

    sub_name = file_stem.split("_")[2][:5]
    parad_name = file_stem.split("_")[2][5:]

    try:
        parad_name_2 = file_stem.split("_")[3]
        parad_name += "_" + parad_name_2

    except:
        parad_name = parad_name
    
    print(sub_name, parad_name)
    
    with open(fl, "rb") as f:
        eeg_fooof_all_subs[sub_name][parad_name] = pkl.load(f)
        f.close()
        

sub01 rest_init
sub01 reward
sub03 rest_init
sub03 reward
sub04 rest_init
sub04 reward
sub07 music_segs
sub07 pink_segs
sub07 rest_end
sub07 rest_init
sub07 rest_mid
sub07 reward
sub09 music_segs
sub09 pink_noise
sub09 pink_segs
sub09 rest_end
sub09 rest_init
sub09 rest_mid
sub09 reward
sub10 music_segs
sub10 pink_noise
sub10 pink_segs
sub10 rest_init
sub10 rest_mid
sub10 reward
sub12 music_segs
sub12 pink_noise
sub12 pink_segs
sub12 rest_end
sub12 rest_init
sub12 rest_mid
sub12 reward


### Plot the Differences in FOOOF between Conditions

Import AAL Atlas

In [11]:
data_dir = path_analysis_data_new + "eeg_source/"

with open(f'{data_dir}AAL_labels.json', 'r') as f:
    aal_labels = json.load(f)

In [12]:
data_dir = path_analysis_data_new + "eeg_source/"
aal_nii_path = f'{data_dir}AAL.nii'   # adjust filename if needed
aal_img = nib.load(aal_nii_path)

aal_data = np.squeeze(np.asanyarray(aal_img.dataobj)).astype(int)
print("AAL Atlas Shape: ", aal_data.shape)

AAL Atlas Shape:  (91, 109, 91)


In [13]:
aal_regions = np.unique(aal_data)
aal_regions = aal_regions[aal_regions != 0]
aal_regions

data_dict = {
    "id": [],
    "regions": [],
    "labels": []
}

with open(path_analysis_data_new + "eeg_source/aal1_for_SPM8/ROI_MNI_V4.txt", "r") as f:
    for line in f:
        if line.strip():
            parts = line.split()  # split by whitespace
            
            data_dict["id"].append(parts[0])
            data_dict["regions"].append(parts[1])
            data_dict["labels"].append(parts[2])

print(data_dict.keys())


aal_regions_str = [str(reg) for reg in aal_regions]

count_reg_similar = 0

for reg in aal_regions_str:
    if reg in data_dict["labels"]:
        count_reg_similar += 1

print("Similar Regions Number: ", count_reg_similar)


label_region_mapping = {}

for indx in range(len(data_dict["labels"])):
    label_region_mapping[int(data_dict["labels"][indx])] = data_dict["regions"][indx]


dict_keys(['id', 'regions', 'labels'])
Similar Regions Number:  116


#### 1. Rest vs. Reward

In [14]:
eeg_fooof_rest_reward = {}


eeg_fooof_rest_reward["sub01"] = {}
eeg_fooof_rest_reward["sub01"]["rest_init"] = eeg_fooof_all_subs["sub01"]["rest_init"].copy()
eeg_fooof_rest_reward["sub01"]["reward"] = eeg_fooof_all_subs["sub01"]["reward"].copy()

eeg_fooof_rest_reward["sub03"] = {}
eeg_fooof_rest_reward["sub03"]["rest_init"] = eeg_fooof_all_subs["sub03"]["rest_init"].copy()
eeg_fooof_rest_reward["sub03"]["reward"] = eeg_fooof_all_subs["sub03"]["reward"].copy()

eeg_fooof_rest_reward["sub04"] = {}
eeg_fooof_rest_reward["sub04"]["rest_init"] = eeg_fooof_all_subs["sub04"]["rest_init"].copy()
eeg_fooof_rest_reward["sub04"]["reward"] = eeg_fooof_all_subs["sub04"]["reward"].copy()

eeg_fooof_rest_reward["sub07"] = {}
eeg_fooof_rest_reward["sub07"]["rest_init"] = eeg_fooof_all_subs["sub07"]["rest_init"].copy()
eeg_fooof_rest_reward["sub07"]["reward"] = eeg_fooof_all_subs["sub07"]["reward"].copy()

eeg_fooof_rest_reward["sub09"] = {} 
eeg_fooof_rest_reward["sub09"]["rest_init"] = eeg_fooof_all_subs["sub09"]["rest_init"].copy()
eeg_fooof_rest_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_rest_reward["sub10"] = {} 
eeg_fooof_rest_reward["sub10"]["rest_init"] = eeg_fooof_all_subs["sub10"]["rest_init"].copy()
eeg_fooof_rest_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_rest_reward["sub12"] = {} 
eeg_fooof_rest_reward["sub12"]["rest_init"] = eeg_fooof_all_subs["sub12"]["rest_init"].copy()
eeg_fooof_rest_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

In [15]:
eeg_fooof_rest_reward_matched = get_matched_legths(eeg_fooof_rest_reward)

Subject: sub01, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 80 epochs
(90, 223, 80) (90, 223, 80)
Subject: sub03, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 77 epochs
(90, 223, 77) (90, 223, 77)
Subject: sub04, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 35 epochs
(90, 223, 35) (90, 223, 35)
Subject: sub07, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 73 epochs
(90, 223, 73) (90, 223, 73)
Subject: sub09, Paradigms: ['rest_init', 'reward']
Truncated array1 from 88 → 67 epochs (dropped last 21)
(90, 223, 67) (90, 223, 67)
Subject: sub10, Paradigms: ['rest_init', 'reward']
Truncated array1 from 48 → 43 epochs (dropped last 5)
(90, 223, 43) (90, 223, 43)
Subject: sub12, Paradigms: ['rest_init', 'reward']
Truncated array1 from 70 → 53 epochs (dropped last 17)
(90, 223, 53) (90, 223, 53)


In [16]:
sub_id = "sub10"

freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]
psds  = eeg_fooof_all_subs["sub12"]["pink_segs"][0]

n_regions = 90

_, rest_init_music_fooof_bands_pvalues_abs, fooof_rest_init_music = get_channel_wise_fooof(freqs, eeg_fooof_rest_reward_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals = get_fdr_p_values(rest_init_music_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)
     #0 for absolute and 1 for relative power
    

sub01
Paradigm 0 rest_init
Paradigm 1 reward
sub03
Paradigm 0 rest_init
Paradigm 1 reward
sub04
Paradigm 0 rest_init
Paradigm 1 reward
sub07
Paradigm 0 rest_init
Paradigm 1 reward
sub09
Paradigm 0 rest_init
Paradigm 1 reward
sub10
Paradigm 0 rest_init
Paradigm 1 reward
sub12
Paradigm 0 rest_init
Paradigm 1 reward


In [17]:
raw_pvals, cluster_list, cluster_means = cluster_and_compare_fooof(fooof_rest_init_music, cond1_name="rest_init",cond2_name="reward",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=0) #0 for absolute and 1 for relative power

✅ Done! Used Functional Network clustering
Number of clusters: 8


In [18]:
def filter_networks(cluster_means, networks_to_keep):
    """
    Return a new dict containing only the specified networks, preserving the
    subject -> condition -> band -> network structure.
    """
    import copy

    out = copy.deepcopy(cluster_means)

    for sub in out:
        for cond in out[sub]:
            for band in out[sub][cond]:
                out[sub][cond][band] = {
                    net: out[sub][cond][band][net]
                    for net in networks_to_keep
                    if net in out[sub][cond][band]
                }

    return out


In [19]:
cluster_means["sub01"]["rest_init"]["delta"].keys()

dict_keys(['Sensorimotor Network', 'Dorsal Attention Network', 'Limbic Network', 'Frontoparietal Network', 'Default Mode Network', 'Ventral Attention Network', 'Visual Network', 'Basal Ganglia Network'])

In [20]:
cluster_means_filtered = filter_networks(
    cluster_means,
    networks_to_keep=NETWORKS
)

In [21]:
plots_path_power = plots_path + "fdr_source/fc_powers_rest_music_cont_all/"

pval_rest_reward = plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="rest_init", condition2="reward", bands=("delta", "theta", "alpha", "beta", "gamma low"),
    alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), legend_label1="Rest", legend_label2="Music Cont.", save_dir=plots_path_power, save=True, subject_order=None, 
    pat_state=pat_state_dict, vertical=True,  FS_PVAL = 20, FS_YLAB = 26, FS_TITLE = 26, show_ylabel=False, fdr_scope="subject")

stack_subject_figures_pdf(folder_path=plots_path_power, stack_mode="vertical", output_path=plots_path_power + "merged_all/src_violin_rest_musiccont.pdf", fig_width=18)

Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_rest_music_cont_all/merged_all/legend_rest_init_vs_reward_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_rest_music_cont_all/merged_all/src_violin_rest_musiccont.pdf


In [22]:
common = dict(cluster_means=cluster_means_filtered, condition1="rest_init", condition2="reward",
              bands=("delta", "theta", "alpha", "beta", "gamma low"),
              alpha_level=0.05, n_bootstrap=1000, save=False, pat_state=pat_state_dict)
p_net = plot_band_power_comparisons(**common, fdr_scope="network")
p_sub = plot_band_power_comparisons(**common, fdr_scope="subject")
changed = 0
for sub in p_net:
    for band in p_net[sub]:
        for net in p_net[sub][band]:
            if (p_net[sub][band][net] < 0.05) != (p_sub[sub][band][net] < 0.05):
                changed += 1
                print(f"{sub} {band:10s} {net:25s} network-FDR p={p_net[sub][band][net]:.3f} -> subject-FDR p={p_sub[sub][band][net]:.3f}")
print(f"{changed} significance decisions changed (rest vs continuous music, periodic)")

Running bootstrap tests …
Done.
Running bootstrap tests …
Done.
sub04 theta      Sensorimotor Network      network-FDR p=0.040 -> subject-FDR p=0.080
1 significance decisions changed (rest vs continuous music, periodic)


In [23]:
mcs_minus_subjects = ["sub04", "sub07", "sub09", "sub10", "sub12"]
networks = NETWORKS

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test_network(
    cluster_means_filtered,
    mcs_minus_subjects,
    condition1="rest_init",
    condition2="reward",
    bands=bands,
    networks=networks,
)

networks = NETWORKS

fdr_corrected_p_vals_arr, sig_mask_arr = build_channel_style_arrays(
    pval_rest_reward,  # your individual-level network p-values dict
    sig_mask,                       # from group_level_bootstrap_test_network
    mcs_minus_subjects,
    bands,
    networks,
)


latex_code = latex_group_vs_individual_table(
    fdr_corrected_p_vals_arr,
    mcs_minus_subjects,
    sig_mask_arr,
    bands,
    comparison_label="Rest vs. Continuous Music",
    mode='start',
    n_channels=len(networks),  # 3, not 128
    caption="Group-level versus individual-level significance across functional networks and frequency bands.",
    label="tab:group_vs_individual_network",
)

print(latex_code)



print("Raw (uncorrected) group-level p-values, rest vs continuous music:")
for band in raw_pvals:
    for net in raw_pvals[band]:
        flag = "  <-- p < 0.05 uncorrected" if raw_pvals[band][net] < 0.05 else ""
        print(f"  {band:10s} {net:25s} p = {raw_pvals[band][net]:.4f}{flag}")

\begin{table}[htbp]
\centering
\caption{Group-level versus individual-level significance across functional networks and frequency bands.}
\label{tab:group_vs_individual_network}

\begin{tabularx}{\textwidth}{@{}l l X X X X@{}}
\toprule
\textbf{Comparison} &
\textbf{Band} &
\textbf{Group-level sig.} &
\textbf{Sig. in $\geq$1 subject} &
\textbf{Sig. in majority} &
\textbf{Sig. in all subjects} \\
\midrule

\multirow{5}{*}{\shortstack[l]{Rest vs.\\Continuous Music}}
& $\delta$ & 0/3 & 3/3 & 0/3 ($\geq$3/5) & 0/3 \\
& $\theta$ & 0/3 & 3/3 & 1/3 ($\geq$3/5) & 0/3 \\
& $\alpha$ & 0/3 & 3/3 & 3/3 ($\geq$3/5) & 0/3 \\
& $\beta$ & 0/3 & 3/3 & 2/3 ($\geq$3/5) & 0/3 \\
& $\gamma$ & 0/3 & 3/3 & 3/3 ($\geq$3/5) & 0/3 \\

\bottomrule
\end{tabularx}
\end{table}

Raw (uncorrected) group-level p-values, rest vs continuous music:
  delta      Sensorimotor Network      p = 0.9670
  delta      Default Mode Network      p = 0.9450
  delta      Frontoparietal Network    p = 0.8210
  theta      Sensorimotor 

In [24]:
# plots_path_power = plots_path + "fdr_source/powers_rest_reward_all/anatomical/"

# raw_pvals, cluster_list, cluster_means = cluster_and_compare_fooof(fooof_rest_init_music, cond1_name="rest_init",cond2_name="reward",
#     mapping_dict=region_to_anatomical_cluster, is_anatomical=True, avg_method='mean', alpha=0.05, abs_rel=1)

# plot_band_power_comparisons(cluster_means=cluster_means, condition1="rest_init", condition2="reward", bands=("delta", "theta", "alpha", "beta", "gamma low"),
#     alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), save_dir=plots_path_power,
# )

# stack_subject_figures(folder_path=plots_path_power, output_path=plots_path_power + "merged_all/merged.png",
#     fig_width=28, row_height=28)

Aperiodic

In [25]:
eeg_fooof_rest_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_rest_reward_matched)
fdr_corrected_p_vals_rest_reward_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_rest_reward_aperiodic, pat_state, n_channels=n_regions)



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward
Paradigm 0:  rest_init
Paradigm 1:  reward


In [26]:

raw_pvals_ap, cluster_list_ap, cluster_means_ap = cluster_and_compare_fooof(eeg_fooof_rest_reward_aperiodic, cond1_name="rest_init",cond2_name="reward",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=2) #2 means aperiodic power, 0 for absolute and 1 for relative power

    
cluster_means_ap_off = {}

for sub in cluster_means_ap:
    cluster_means_ap_off[sub] = {}

    for parad in cluster_means_ap[sub]:
        cluster_means_ap_off[sub][parad] = {}
        cluster_means_ap_off[sub][parad]["Exponent"] = {}
        cluster_means_ap_off[sub][parad]["Offset"] = {}

        for net in cluster_means_ap[sub][parad]["Offset"]:
            cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 10)
            cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])
        
            
            # if sub == "sub03":
            #     cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 20)
            #     cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])

            # # elif sub == "sub02":
            # #     cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] - 10)
            # #     cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])

            # else:
            #     cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 10)
            #     cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])



cluster_means_filtered = filter_networks(
    cluster_means_ap_off,
    networks_to_keep=NETWORKS
)


plots_path_power = plots_path + "fdr_source/fc_powers_rest_music_cont_all/ap/"

save_name = ""

plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="rest_init", condition2="reward", bands=['Offset', 'Exponent'],
    alpha_level=0.05, n_bootstrap=1000, figsize=(3, 5), save=True, legend_label1="Rest", legend_label2="Music Cont.", save_dir=plots_path_power, 
    power_unit=r"Aperiodic Parameters (a.u.)", save_tag=save_name, subject_order=None, pat_state=pat_state_dict, show_legend=False, vertical=True,
    FS_PVAL = 20, FS_YLAB = 26, FS_TITLE = 26, show_ylabel=False, fdr_scope="subject") #FS_PVAL = 18, FS_YLAB = 12, FS_BAND = 22, FS_TITLE = 22)




stack_subject_figures_pdf(folder_path=plots_path_power, stack_mode="vertical", 
                          output_path=plots_path_power + "merged_all/src_aperiodic_violin_rest_musiccont.pdf", fig_width=9)

✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_rest_music_cont_all/ap/merged_all/legend_rest_init_vs_reward_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_rest_music_cont_all/ap/merged_all/src_aperiodic_violin_rest_musiccont.pdf


In [27]:
# common = dict(cluster_means=cluster_means_filtered, condition1="rest_init", condition2="reward",
#               bands=['Offset', 'Exponent'],
#               alpha_level=0.05, n_bootstrap=1000, save=False, pat_state=pat_state_dict)

# p_net = plot_band_power_comparisons(**common, fdr_scope="network")
# p_sub = plot_band_power_comparisons(**common, fdr_scope="subject")
# changed = 0
# for sub in p_net:
#     for band in p_net[sub]:
#         for net in p_net[sub][band]:
#             if (p_net[sub][band][net] < 0.05) != (p_sub[sub][band][net] < 0.05):
#                 changed += 1
#                 print(f"{sub} {band:10s} {net:25s} network-FDR p={p_net[sub][band][net]:.3f} -> subject-FDR p={p_sub[sub][band][net]:.3f}")
# print(f"{changed} significance decisions changed (rest vs continuous music, aperiodic)")

#### Other Comparisons

In [28]:
def split_halves(entry):
    psd, ap, per, fr = entry
    h = per.shape[2] // 2
    return ([psd[..., :h],      ap[:, :h, :],      per[..., :h],      fr],
            [psd[..., h:2 * h], ap[:, h:2 * h, :], per[..., h:2 * h], fr])

def make_split(fooof_all, cond):
    return {sub: dict(zip(["half1", "half2"], split_halves(fooof_all[sub][cond])))
            for sub in fooof_all if cond in fooof_all[sub]}

def truncate_pair(pair, n_per_sub):
    return {sub: {c: [psd[..., :n_per_sub[sub]], ap[:, :n_per_sub[sub], :], per[..., :n_per_sub[sub]], fr]
                  for c, (psd, ap, per, fr) in d.items()}
            for sub, d in pair.items()}

bands_per = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 12), "beta": (12, 30), "gamma low": (30, 45)}
nets = NETWORKS
abbr = NET_ABBR

def run_network_tests(pair, c1, c2, fdr_scope="subject"):
    _, _, fb = get_channel_wise_fooof(freqs, pair, bands_per, channels_n=90)
    _, _, cm = cluster_and_compare_fooof(fb, cond1_name=c1, cond2_name=c2, mapping_dict=region_to_network,
                                         is_anatomical=False, avg_method="mean", abs_rel=0)
    _, _, cm_ap = cluster_and_compare_fooof(get_fooof_aperiodic(pair), cond1_name=c1, cond2_name=c2,
                                            mapping_dict=region_to_network, is_anatomical=False,
                                            avg_method="mean", abs_rel=2)
    p_per, *_ = compute_bootstrap_pvals(filter_networks(cm, nets), c1, c2, list(bands_per),
                                        n_bootstrap=1000, fdr_scope=fdr_scope)
    p_ap, *_ = compute_bootstrap_pvals(filter_networks(cm_ap, nets), c1, c2, ["Offset", "Exponent"],
                                       n_bootstrap=1000, fdr_scope=fdr_scope)
    return p_per, p_ap

split_music = make_split(eeg_fooof_all_subs, "reward")
split_rest  = make_split(eeg_fooof_all_subs, "rest_init")
n_rest_half = {sub: split_rest[sub]["half1"][2].shape[2] for sub in split_rest}
rm_trunc    = truncate_pair(eeg_fooof_rest_reward_matched, n_rest_half)

results_net = {
    "music half1 vs half2":          run_network_tests(split_music, "half1", "half2"),
    "rest vs music (full N)":        run_network_tests(eeg_fooof_rest_reward_matched, "rest_init", "reward"),
    "rest half1 vs half2":           run_network_tests(split_rest, "half1", "half2"),
    "rest vs music (N = rest half)": run_network_tests(rm_trunc, "rest_init", "reward"),
}

for name, (p_per, p_ap) in results_net.items():
    print(f"\n=== {name} === (significant band-network pairs, joint FDR per subject)")
    for sub in sorted(p_per):
        sig = [f"{b}-{abbr[n]}" for p in (p_per, p_ap) for b in p.get(sub, {})
               for n in p[sub][b] if p[sub][b][n] < 0.05]
        print(f"  {sub}: {len(sig):2d}  {', '.join(sig)}")

sub01
Paradigm 0 half1
Paradigm 1 half2
sub03
Paradigm 0 half1
Paradigm 1 half2
sub04
Paradigm 0 half1
Paradigm 1 half2
sub07
Paradigm 0 half1
Paradigm 1 half2
sub09
Paradigm 0 half1
Paradigm 1 half2
sub10
Paradigm 0 half1
Paradigm 1 half2
sub12
Paradigm 0 half1
Paradigm 1 half2
✅ Done! Used Functional Network clustering
Number of clusters: 8
✅ Done! Used Functional Network clustering
Number of clusters: 8
sub01
Paradigm 0 rest_init
Paradigm 1 reward
sub03
Paradigm 0 rest_init
Paradigm 1 reward
sub04
Paradigm 0 rest_init
Paradigm 1 reward
sub07
Paradigm 0 rest_init
Paradigm 1 reward
sub09
Paradigm 0 rest_init
Paradigm 1 reward
sub10
Paradigm 0 rest_init
Paradigm 1 reward
sub12
Paradigm 0 rest_init
Paradigm 1 reward
✅ Done! Used Functional Network clustering
Number of clusters: 8
✅ Done! Used Functional Network clustering
Number of clusters: 8
sub01
Paradigm 0 half1
Paradigm 1 half2
sub03
Paradigm 0 half1
Paradigm 1 half2
sub04
Paradigm 0 half1
Paradigm 1 half2
sub07
Paradigm 0 half1
Pa

In [29]:
# # channel notebook
# print(split_half_latex_table(results, "channel", ["delta", "theta", "alpha", "beta", "gamma low"], 128,
#                              "tab:split_half_channel"))
# # region notebook
# print(split_half_latex_table(results_src, "region", ["delta", "theta", "alpha", "beta", "gamma low"], 90,
#                              "tab:split_half_region"))
# network notebook
print(split_half_latex_table(results_net, "network", list(bands_per), 3, "tab:split_half_network",
                             pat_state=pat_state_dict))

\begingroup
\small
\setlength{\tabcolsep}{4pt}
\begin{xltabular}{\textwidth}{@{}l X r r r r r r r@{}}
\caption{Split-half control at the network level. Number of significant networks (of 3) per subject, frequency band, and aperiodic parameter, for the initial rest versus continuous music comparison and for the within-recording split-half comparisons (H1, H2: first and second half of the recording). The rest versus music comparison is shown with all epochs and with its epoch count reduced to that of the rest halves. Bold: rest-versus-music count exceeds the corresponding split-half count in both pairings.}
\label{tab:split_half_network}\\
\toprule
\textbf{Subject} & \textbf{Comparison} & $\delta$ & $\theta$ & $\alpha$ & $\beta$ & $\gamma$ & Offset & Exp. \\
\midrule
\endhead
\multirow{4}{*}{SUB1 (UWS)} & Initial rest vs music & \textbf{1} & \textbf{1} & \textbf{3} & \textbf{3} & \textbf{2} & 1 & \textbf{3} \\
 & Music H1 vs H2 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & Rest vs music (N = rest ha

In [30]:
def network_comparison(subs, c1, c2, lab1, lab2, folder, tag):
    """One network-level comparison, periodic and aperiodic, with the same settings as section 3."""
    pair = get_matched_legths({s: {c1: eeg_fooof_all_subs[s][c1].copy(),
                                   c2: eeg_fooof_all_subs[s][c2].copy()} for s in subs})

    # ---- periodic ----
    _, _, fb = get_channel_wise_fooof(freqs, pair, bands_per, channels_n=90)
    _, _, cm = cluster_and_compare_fooof(fb, cond1_name=c1, cond2_name=c2, mapping_dict=region_to_network,
                                         is_anatomical=False, avg_method="mean", alpha=0.05, abs_rel=0)
    cm = filter_networks(cm, nets)
    out = plots_path + f"fdr_source/{folder}/"
    p_per = plot_band_power_comparisons(
        cluster_means=cm, condition1=c1, condition2=c2, bands=tuple(bands_per),
        alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), legend_label1=lab1, legend_label2=lab2,
        save=True, save_dir=out, subject_order=None, pat_state=pat_state_dict,
        FS_PVAL=18, FS_YLAB=20, FS_BAND=20, FS_TITLE=20, fdr_scope="subject")
    stack_subject_figures_pdf(folder_path=out, output_path=out + f"merged_all/src_violin_{tag}.pdf")
    plt.close("all")

    # ---- aperiodic (offset shifted by +10 for display only) ----
    _, _, cm_ap = cluster_and_compare_fooof(get_fooof_aperiodic(pair), cond1_name=c1, cond2_name=c2,
                                            mapping_dict=region_to_network, is_anatomical=False,
                                            avg_method="mean", alpha=0.05, abs_rel=2)
    cm_ap = filter_networks(
        {sub: {cond: {"Offset":   {n: np.asarray(v["Offset"][n]) + 10 for n in v["Offset"]},
                      "Exponent": {n: np.asarray(v["Exponent"][n]) for n in v["Exponent"]}}
               for cond, v in d.items()} for sub, d in cm_ap.items()}, nets)
    out_ap = out + "ap/"
    p_ap = plot_band_power_comparisons(
        cluster_means=cm_ap, condition1=c1, condition2=c2, bands=["Offset", "Exponent"],
        alpha_level=0.05, n_bootstrap=1000, figsize=(3, 5), legend_label1=lab1, legend_label2=lab2,
        save_dir=out_ap, show_legend=False, power_unit=r"Aperiodic Parameters (a.u.)", save=True,
        save_tag="", pat_state=pat_state_dict, FS_PVAL=18, FS_YLAB=20, FS_BAND=20, FS_TITLE=20,
        fdr_scope="subject")
    stack_subject_figures_pdf(folder_path=out_ap, output_path=out_ap + f"merged_all/src_aperiodic_violin_{tag}.pdf")
    plt.close("all")

    for name, p in (("periodic", p_per), ("aperiodic", p_ap)):
        for sub in sorted(p):
            sig = [f"{b}-{abbr[n]}" for b in p[sub] for n in p[sub][b] if p[sub][b][n] < 0.05]
            print(f"{tag} {name} {sub}: {len(sig):2d}  {', '.join(sig)}")
    return pair, cm, p_per, p_ap


# 1) intermediate rest vs segmented music (Sub4-Sub7)
pair_rm_music, cm_rm_music, p_rm_music, p_rm_music_ap = network_comparison(
    ["sub07", "sub09", "sub10", "sub12"], "rest_mid", "music_segs", "Rest (Intermediate)", "Music Segs",
    "fc_powers_rest_mid_music_segs_all", "restmid_musicsegs")

# 2) intermediate rest vs segmented pink noise (Sub4-Sub7)
pair_rm_pink, cm_rm_pink, p_rm_pink, p_rm_pink_ap = network_comparison(
    ["sub07", "sub09", "sub10", "sub12"], "rest_mid", "pink_segs", "Rest (Intermediate)", "Pink Noise Segs",
    "fc_powers_rest_mid_pink_segs_all", "restmid_pinksegs")

Subject: sub07, Paradigms: ['rest_mid', 'music_segs']
Truncated array2 from 125 → 36 epochs (dropped last 89)
(90, 223, 36) (90, 223, 36)
Subject: sub09, Paradigms: ['rest_mid', 'music_segs']
Truncated array2 from 93 → 44 epochs (dropped last 49)
(90, 223, 44) (90, 223, 44)
Subject: sub10, Paradigms: ['rest_mid', 'music_segs']
Truncated array2 from 68 → 23 epochs (dropped last 45)
(90, 223, 23) (90, 223, 23)
Subject: sub12, Paradigms: ['rest_mid', 'music_segs']
Truncated array2 from 82 → 39 epochs (dropped last 43)
(90, 223, 39) (90, 223, 39)
sub07
Paradigm 0 rest_mid
Paradigm 1 music_segs
sub09
Paradigm 0 rest_mid
Paradigm 1 music_segs
sub10
Paradigm 0 rest_mid
Paradigm 1 music_segs
sub12
Paradigm 0 rest_mid
Paradigm 1 music_segs
✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_rest_mid_music_segs_all/merged

In [31]:
# ============================================================
# FULL NETWORK-LEVEL RESULTS PRINTOUT (self-contained)
# Recomputes all p-values with the same settings/seed as the figures
# (fdr_scope="subject", n_bootstrap=1000, default rng_seed) and prints
# p-value and sign of the mean difference (condition2 - condition1).
# Also writes everything to a text file next to the plots.
# ============================================================
import io as _io

freqs_s   = eeg_fooof_all_subs["sub12"]["pink_segs"][3]
bands_s   = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 12), "beta": (12, 30), "gamma low": (30, 45)}
nets_s    = NETWORKS
abbr_s    = NET_ABBR
band_lab  = {"delta": "d", "theta": "t", "alpha": "a", "beta": "b", "gamma low": "g",
             "Offset": "Off", "Exponent": "Exp"}

def _pair(subs, c1, c2):
    return get_matched_legths({s: {c1: eeg_fooof_all_subs[s][c1].copy(),
                                   c2: eeg_fooof_all_subs[s][c2].copy()} for s in subs})

def _nettest(pair, c1, c2):
    _, _, fb = get_channel_wise_fooof(freqs_s, pair, bands_s, channels_n=90)
    _, _, cm = cluster_and_compare_fooof(fb, cond1_name=c1, cond2_name=c2, mapping_dict=region_to_network,
                                         is_anatomical=False, avg_method="mean", abs_rel=0)
    _, _, cm_ap = cluster_and_compare_fooof(get_fooof_aperiodic(pair), cond1_name=c1, cond2_name=c2,
                                            mapping_dict=region_to_network, is_anatomical=False,
                                            avg_method="mean", abs_rel=2)
    p1, _, _, o1 = compute_bootstrap_pvals(filter_networks(cm, nets_s), c1, c2, list(bands_s),
                                           n_bootstrap=1000, fdr_scope="subject")
    p2, _, _, o2 = compute_bootstrap_pvals(filter_networks(cm_ap, nets_s), c1, c2, ["Offset", "Exponent"],
                                           n_bootstrap=1000, fdr_scope="subject")
    n_ep = {s: pair[s][c1][2].shape[2] for s in pair}
    return (p1, o1), (p2, o2), n_ep

def _report(buf, name, pair, c1, c2):
    (p1, o1), (p2, o2), n_ep = _nettest(pair, c1, c2)
    print(f"\n=== {name}  [{c1} -> {c2}] ===", file=buf)
    print("epochs per condition:", {s: n_ep[s] for s in sorted(n_ep)}, file=buf)
    for comp, (pv, ob) in (("periodic", (p1, o1)), ("aperiodic", (p2, o2))):
        for s in sorted(pv):
            for net in nets_s:
                cells = []
                for b in pv[s]:
                    if net not in pv[s][b]:
                        continue
                    p = pv[s][b][net]
                    sgn = "+" if ob[s][b][net] > 0 else "-"
                    star = "*" if p < 0.05 else " "
                    cells.append(f"{band_lab[b]}:{p:.3f}({sgn}){star}")
                print(f"  {s} {comp:9s} {abbr_s[net]:3s}  " + "  ".join(cells), file=buf)
    sig = {s: [f"{comp[0]}:{band_lab[b]}-{abbr_s[n]}({'+' if ob[s][b][n] > 0 else '-'})"
               for comp, (pv, ob) in (("per", (p1, o1)), ("ap", (p2, o2)))
               for b in pv.get(s, {}) for n in pv[s][b] if pv[s][b][n] < 0.05]
           for s in sorted(p1)}
    for s, lst in sig.items():
        print(f"  SIG {s}: {len(lst):2d}  {', '.join(lst)}", file=buf)

buf = _io.StringIO()
print("NETWORK-LEVEL RESULTS SUMMARY  (p FDR-corrected per subject across all band-network tests;",
      "sign = mean(condition2) - mean(condition1); * = p < 0.05)", file=buf)

subs_all  = ["sub01", "sub03", "sub04", "sub07", "sub09", "sub10", "sub12"]
subs_seg  = ["sub07", "sub09", "sub10", "sub12"]
subs_cont = ["sub09", "sub10", "sub12"]

rr = _pair(subs_all, "rest_init", "reward")
_report(buf, "1. initial rest vs continuous music (full N)", rr, "rest_init", "reward")
_report(buf, "2. segmented pink noise vs segmented music", _pair(subs_seg, "pink_segs", "music_segs"), "pink_segs", "music_segs")
_report(buf, "3. continuous pink noise vs continuous music", _pair(subs_cont, "pink_noise", "reward"), "pink_noise", "reward")
_report(buf, "4. intermediate rest vs segmented music", _pair(subs_seg, "rest_mid", "music_segs"), "rest_mid", "music_segs")
_report(buf, "5. intermediate rest vs segmented pink noise", _pair(subs_seg, "rest_mid", "pink_segs"), "rest_mid", "pink_segs")

# split-half controls (rest and continuous music)
split_music_s = make_split(eeg_fooof_all_subs, "reward")
split_rest_s  = make_split(eeg_fooof_all_subs, "rest_init")
n_rest_half_s = {s: split_rest_s[s]["half1"][2].shape[2] for s in split_rest_s}
rm_trunc_s    = truncate_pair(rr, n_rest_half_s)
_report(buf, "6. SPLIT music half1 vs half2", split_music_s, "half1", "half2")
_report(buf, "7. SPLIT rest half1 vs half2", split_rest_s, "half1", "half2")
_report(buf, "8. rest vs music (N = rest half)", rm_trunc_s, "rest_init", "reward")

text = buf.getvalue()
print(text)
out_file = plots_path + "network_level_results_summary.txt"
with open(out_file, "w", encoding="utf-8") as f:
    f.write(text)
print("\nsaved ->", out_file)

Subject: sub01, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 80 epochs
(90, 223, 80) (90, 223, 80)
Subject: sub03, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 77 epochs
(90, 223, 77) (90, 223, 77)
Subject: sub04, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 35 epochs
(90, 223, 35) (90, 223, 35)
Subject: sub07, Paradigms: ['rest_init', 'reward']
✓ Lengths already match: 73 epochs
(90, 223, 73) (90, 223, 73)
Subject: sub09, Paradigms: ['rest_init', 'reward']
Truncated array1 from 88 → 67 epochs (dropped last 21)
(90, 223, 67) (90, 223, 67)
Subject: sub10, Paradigms: ['rest_init', 'reward']
Truncated array1 from 48 → 43 epochs (dropped last 5)
(90, 223, 43) (90, 223, 43)
Subject: sub12, Paradigms: ['rest_init', 'reward']
Truncated array1 from 70 → 53 epochs (dropped last 17)
(90, 223, 53) (90, 223, 53)
sub01
Paradigm 0 rest_init
Paradigm 1 reward
sub03
Paradigm 0 rest_init
Paradigm 1 reward
sub04
Paradigm 0 rest_init
Paradigm 1 reward
sub07
Pa

#### 2. Music Segs vs. Pink Segs

In [32]:
eeg_fooof_music_pink_segs = {}


eeg_fooof_music_pink_segs["sub07"] = {}
eeg_fooof_music_pink_segs["sub07"]["pink_segs"] = eeg_fooof_all_subs["sub07"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub07"]["music_segs"] = eeg_fooof_all_subs["sub07"]["music_segs"].copy()


eeg_fooof_music_pink_segs["sub09"] = {} 
eeg_fooof_music_pink_segs["sub09"]["pink_segs"] = eeg_fooof_all_subs["sub09"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub09"]["music_segs"] = eeg_fooof_all_subs["sub09"]["music_segs"].copy()


eeg_fooof_music_pink_segs["sub10"] = {} 
eeg_fooof_music_pink_segs["sub10"]["pink_segs"] = eeg_fooof_all_subs["sub10"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub10"]["music_segs"] = eeg_fooof_all_subs["sub10"]["music_segs"].copy()


eeg_fooof_music_pink_segs["sub12"] = {} 
eeg_fooof_music_pink_segs["sub12"]["pink_segs"] = eeg_fooof_all_subs["sub12"]["pink_segs"].copy()
eeg_fooof_music_pink_segs["sub12"]["music_segs"] = eeg_fooof_all_subs["sub12"]["music_segs"].copy()


eeg_fooof_music_pink_segs_matched = get_matched_legths(eeg_fooof_music_pink_segs)

Subject: sub07, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 128 → 125 epochs (dropped last 3)
(90, 223, 125) (90, 223, 125)
Subject: sub09, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 94 → 93 epochs (dropped last 1)
(90, 223, 93) (90, 223, 93)
Subject: sub10, Paradigms: ['pink_segs', 'music_segs']
Truncated array2 from 68 → 35 epochs (dropped last 33)
(90, 223, 35) (90, 223, 35)
Subject: sub12, Paradigms: ['pink_segs', 'music_segs']
Truncated array1 from 83 → 82 epochs (dropped last 1)
(90, 223, 82) (90, 223, 82)


In [33]:
sub_id = "sub10"

freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]
psds  = eeg_fooof_all_subs["sub12"]["pink_segs"][0]

n_regions = 90

_, music_pink_seg_fooof_bands_pvalues_abs, fooof_music_pink_segs = get_channel_wise_fooof(freqs, eeg_fooof_music_pink_segs_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_music_pink_segs = get_fdr_p_values(music_pink_seg_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)


sub07
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub09
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub10
Paradigm 0 pink_segs
Paradigm 1 music_segs
sub12
Paradigm 0 pink_segs
Paradigm 1 music_segs


In [34]:
plots_path_power = plots_path + "fdr_source/fc_powers_music_segs_pink_segs_all/"

raw_pvals, cluster_list, cluster_means = cluster_and_compare_fooof(fooof_music_pink_segs, cond1_name="pink_segs",cond2_name="music_segs",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=0)



cluster_means_filtered = filter_networks(
    cluster_means,
    networks_to_keep=NETWORKS
)


pvals_music_pink_segs = plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="pink_segs", condition2="music_segs", bands=("delta", "theta", "alpha", "beta", "gamma low"),
    alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), legend_label1="Pink N. Segs", legend_label2="Music Segs", save=True, save_dir=plots_path_power, subject_order=None, 
    pat_state=pat_state_dict, FS_PVAL = 18, FS_YLAB = 20, FS_BAND = 20, FS_TITLE = 20, fdr_scope="subject"      
)

stack_subject_figures_pdf(folder_path=plots_path_power, output_path=plots_path_power + "merged_all/src_violin_musicsegs_pinksegs.pdf")

✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_music_segs_pink_segs_all/merged_all/legend_pink_segs_vs_music_segs_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_music_segs_pink_segs_all/merged_all/src_violin_musicsegs_pinksegs.pdf


In [35]:
common = dict(cluster_means=cluster_means_filtered, condition1="pink_segs", condition2="music_segs",
              bands=("delta", "theta", "alpha", "beta", "gamma low"),
              alpha_level=0.05, n_bootstrap=1000, save=False, pat_state=pat_state_dict)
p_net = plot_band_power_comparisons(**common, fdr_scope="network")
p_sub = plot_band_power_comparisons(**common, fdr_scope="subject")
changed = 0
for sub in p_net:
    for band in p_net[sub]:
        for net in p_net[sub][band]:
            if (p_net[sub][band][net] < 0.05) != (p_sub[sub][band][net] < 0.05):
                changed += 1
                print(f"{sub} {band:10s} {net:25s} network-FDR p={p_net[sub][band][net]:.3f} -> subject-FDR p={p_sub[sub][band][net]:.3f}")
print(f"{changed} significance decisions changed (segmented music vs pink noise, periodic)")

Running bootstrap tests …
Done.
Running bootstrap tests …
Done.
sub10 delta      Frontoparietal Network    network-FDR p=0.052 -> subject-FDR p=0.039
1 significance decisions changed (segmented music vs pink noise, periodic)


In [36]:
mcs_minus_subjects = ["sub07", "sub09", "sub10", "sub12"]
networks = NETWORKS

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test_network(
    cluster_means_filtered,
    mcs_minus_subjects,
    condition1="pink_segs",
    condition2="music_segs",
    bands=bands,
    networks=networks,
)

networks = NETWORKS

fdr_corrected_p_vals_arr, sig_mask_arr = build_channel_style_arrays(
    pvals_music_pink_segs,  # your individual-level network p-values dict
    sig_mask,                       # from group_level_bootstrap_test_network
    mcs_minus_subjects,
    bands,
    networks,
)

latex_code = latex_group_vs_individual_table(
    fdr_corrected_p_vals_arr,
    mcs_minus_subjects,
    sig_mask_arr,
    bands,
    comparison_label="Music Segs. vs. Pink Noise Segs.",
    mode='append',
    n_channels=len(networks),  # 3, not 128
    
)

print(latex_code)


print("Raw (uncorrected) group-level p-values, segmented music vs segmented pink noise:")
for band in raw_pvals:
    for net in raw_pvals[band]:
        flag = "  <-- p < 0.05 uncorrected" if raw_pvals[band][net] < 0.05 else ""
        print(f"  {band:10s} {net:25s} p = {raw_pvals[band][net]:.4f}{flag}")

\midrule

\multirow{5}{*}{\shortstack[l]{Music Segs. vs.\\Pink Noise Segs.}}
& $\delta$ & 0/3 & 3/3 & 0/3 ($\geq$2/4) & 0/3 \\
& $\theta$ & 0/3 & 1/3 & 0/3 ($\geq$2/4) & 0/3 \\
& $\alpha$ & 0/3 & 1/3 & 0/3 ($\geq$2/4) & 0/3 \\
& $\beta$ & 0/3 & 0/3 & 0/3 ($\geq$2/4) & 0/3 \\
& $\gamma$ & 0/3 & 0/3 & 0/3 ($\geq$2/4) & 0/3 \\


Raw (uncorrected) group-level p-values, segmented music vs segmented pink noise:
  delta      Sensorimotor Network      p = 0.2470
  delta      Default Mode Network      p = 0.1630
  delta      Frontoparietal Network    p = 0.1100
  theta      Sensorimotor Network      p = 0.0510
  theta      Default Mode Network      p = 0.1291
  theta      Frontoparietal Network    p = 0.2490
  alpha      Sensorimotor Network      p = 0.4440
  alpha      Default Mode Network      p = 0.4420
  alpha      Frontoparietal Network    p = 0.4770
  beta       Sensorimotor Network      p = 0.0540
  beta       Default Mode Network      p = 0.6320
  beta       Frontoparietal Network    p 

Aperiodic

In [37]:
eeg_fooof_music_pink_segs_aperiodic = get_fooof_aperiodic(eeg_fooof_music_pink_segs_matched)

fdr_corrected_p_vals_music_pink_segs_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_music_pink_segs_aperiodic, pat_state, n_channels=n_regions)


raw_pvals_ap, cluster_list_ap, cluster_means_ap = cluster_and_compare_fooof(eeg_fooof_music_pink_segs_aperiodic, cond1_name="pink_segs",cond2_name="music_segs",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=2)


cluster_means_ap_off = {}

for sub in cluster_means_ap:
    cluster_means_ap_off[sub] = {}

    for parad in cluster_means_ap[sub]:
        cluster_means_ap_off[sub][parad] = {}
        cluster_means_ap_off[sub][parad]["Exponent"] = {}
        cluster_means_ap_off[sub][parad]["Offset"] = {}

        for net in cluster_means_ap[sub][parad]["Offset"]:
            cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 10)
            cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])
            # if sub == "sub03":
            #     cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 20)
            #     cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])



cluster_means_filtered = filter_networks(
    cluster_means_ap_off,
    networks_to_keep=NETWORKS
)


plots_path_power = plots_path + "fdr_source/fc_powers_music_segs_pink_segs_all/ap/"

save_name = ""

p_Val_music_pink_seg = plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="pink_segs", condition2="music_segs", bands=['Offset', 'Exponent'],
    alpha_level=0.05, n_bootstrap=1000, figsize=(3, 5), legend_label1="Pink N. Segs", legend_label2="Music Segs", power_unit=r"Aperiodic Parameters (a.u.)", 
    show_legend=False, save_dir=plots_path_power, save=True, save_tag=save_name, pat_state=pat_state_dict, FS_PVAL = 18, FS_YLAB = 20, FS_BAND = 20, FS_TITLE = 20, fdr_scope="subject")


stack_subject_figures_pdf(folder_path=plots_path_power, output_path=plots_path_power + "merged_all/src_aperiodic_violin_musicsegs_pinksegs.pdf")

Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
Paradigm 0:  pink_segs
Paradigm 1:  music_segs
✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_music_segs_pink_segs_all/ap/merged_all/legend_pink_segs_vs_music_segs_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_music_segs_pink_segs_all/ap/merged_all/src_aperiodic_violin_musicsegs_pinksegs.pdf


In [38]:
common = dict(cluster_means=cluster_means_filtered, condition1="pink_segs", condition2="music_segs",
              bands=['Offset', 'Exponent'],
              alpha_level=0.05, n_bootstrap=1000, save=False, pat_state=pat_state_dict)
p_net = plot_band_power_comparisons(**common, fdr_scope="network")
p_sub = plot_band_power_comparisons(**common, fdr_scope="subject")
changed = 0
for sub in p_net:
    for band in p_net[sub]:
        for net in p_net[sub][band]:
            if (p_net[sub][band][net] < 0.05) != (p_sub[sub][band][net] < 0.05):
                changed += 1
                print(f"{sub} {band:10s} {net:25s} network-FDR p={p_net[sub][band][net]:.3f} -> subject-FDR p={p_sub[sub][band][net]:.3f}")
print(f"{changed} significance decisions changed (segmented music vs pink noise, aperiodic)")

Running bootstrap tests …
Done.
Running bootstrap tests …
Done.
0 significance decisions changed (segmented music vs pink noise, aperiodic)


#### 3. Reward vs. Pink Noise

In [39]:
eeg_fooof_pink_noise_reward = {}

eeg_fooof_pink_noise_reward["sub09"] = {} 
eeg_fooof_pink_noise_reward["sub09"]["pink_noise"] = eeg_fooof_all_subs["sub09"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub09"]["reward"] = eeg_fooof_all_subs["sub09"]["reward"].copy()

eeg_fooof_pink_noise_reward["sub10"] = {} 
eeg_fooof_pink_noise_reward["sub10"]["pink_noise"] = eeg_fooof_all_subs["sub10"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub10"]["reward"] = eeg_fooof_all_subs["sub10"]["reward"].copy()

eeg_fooof_pink_noise_reward["sub12"] = {}
eeg_fooof_pink_noise_reward["sub12"]["pink_noise"] = eeg_fooof_all_subs["sub12"]["pink_noise"].copy()
eeg_fooof_pink_noise_reward["sub12"]["reward"] = eeg_fooof_all_subs["sub12"]["reward"].copy()

eeg_fooof_pink_noise_reward_matched = get_matched_legths(eeg_fooof_pink_noise_reward)

Subject: sub09, Paradigms: ['pink_noise', 'reward']
Truncated array2 from 67 → 51 epochs (dropped last 16)
(90, 223, 51) (90, 223, 51)
Subject: sub10, Paradigms: ['pink_noise', 'reward']
Truncated array2 from 43 → 36 epochs (dropped last 7)
(90, 223, 36) (90, 223, 36)
Subject: sub12, Paradigms: ['pink_noise', 'reward']
Truncated array1 from 58 → 53 epochs (dropped last 5)
(90, 223, 53) (90, 223, 53)


In [40]:
sub_id = "sub10"

freqs = eeg_fooof_all_subs["sub12"]["pink_segs"][3]
psds  = eeg_fooof_all_subs["sub12"]["pink_segs"][0]

n_regions = 90
    
_, pink_noise_reward_fooof_bands_pvalues_abs, fooof_music_pink_noise= get_channel_wise_fooof(freqs, eeg_fooof_pink_noise_reward_matched, bands, channels_n=n_regions)
fdr_corrected_p_vals_pink_noise_reward = get_fdr_p_values(pink_noise_reward_fooof_bands_pvalues_abs, p_vals_per_band=n_regions)

sub09
Paradigm 0 pink_noise
Paradigm 1 reward
sub10
Paradigm 0 pink_noise
Paradigm 1 reward
sub12
Paradigm 0 pink_noise
Paradigm 1 reward


In [41]:
plots_path_power = plots_path + "fdr_source/fc_powers_music_cont_pink_noise_cont_all/"

raw_pvals, cluster_list, cluster_means = cluster_and_compare_fooof(fooof_music_pink_noise, cond1_name="pink_noise",cond2_name="reward",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=0)


cluster_means_filtered = filter_networks(
    cluster_means,
    networks_to_keep=NETWORKS
)

pvals_pink_reward = plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="pink_noise", condition2="reward", bands=("delta", "theta", "alpha", "beta", "gamma low"),
    alpha_level=0.05, n_bootstrap=1000, figsize=(6, 5), legend_label1="Pink Noise Cont.", legend_label2="Music Cont.", save=True, save_dir=plots_path_power, 
    subject_order=None, pat_state=pat_state_dict, FS_PVAL = 18, FS_YLAB = 20, FS_BAND = 20, FS_TITLE = 20, fdr_scope="subject"
)

stack_subject_figures_pdf(folder_path=plots_path_power, output_path=plots_path_power + "merged_all/src_violin_pinkcont_musiccont.pdf" )

✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_music_cont_pink_noise_cont_all/merged_all/legend_pink_noise_vs_reward_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_music_cont_pink_noise_cont_all/merged_all/src_violin_pinkcont_musiccont.pdf


In [42]:
common = dict(cluster_means=cluster_means_filtered, condition1="pink_noise", condition2="reward",
              bands=("delta", "theta", "alpha", "beta", "gamma low"),
              alpha_level=0.05, n_bootstrap=1000, save=False, pat_state=pat_state_dict)
p_net = plot_band_power_comparisons(**common, fdr_scope="network")
p_sub = plot_band_power_comparisons(**common, fdr_scope="subject")
changed = 0
for sub in p_net:
    for band in p_net[sub]:
        for net in p_net[sub][band]:
            if (p_net[sub][band][net] < 0.05) != (p_sub[sub][band][net] < 0.05):
                changed += 1
                print(f"{sub} {band:10s} {net:25s} network-FDR p={p_net[sub][band][net]:.3f} -> subject-FDR p={p_sub[sub][band][net]:.3f}")
print(f"{changed} significance decisions changed (continuous music vs pink noise, periodic)")

Running bootstrap tests …
Done.
Running bootstrap tests …
Done.
sub10 theta      Default Mode Network      network-FDR p=0.032 -> subject-FDR p=0.065
sub10 gamma low  Sensorimotor Network      network-FDR p=0.045 -> subject-FDR p=0.065
sub10 gamma low  Default Mode Network      network-FDR p=0.032 -> subject-FDR p=0.065
3 significance decisions changed (continuous music vs pink noise, periodic)


In [43]:
mcs_minus_subjects = ["sub09", "sub10", "sub12"]
networks = NETWORKS

raw_pvals, fdr_pvals, sig_mask = group_level_bootstrap_test_network(
    cluster_means_filtered,
    mcs_minus_subjects,
    condition1="pink_noise",
    condition2="reward",
    bands=bands,
    networks=networks,
)

networks = NETWORKS

fdr_corrected_p_vals_arr, sig_mask_arr = build_channel_style_arrays(
    pvals_pink_reward,  # your individual-level network p-values dict
    sig_mask,                       # from group_level_bootstrap_test_network
    mcs_minus_subjects,
    bands,
    networks,
)

latex_code = latex_group_vs_individual_table(
    fdr_corrected_p_vals_arr,
    mcs_minus_subjects,
    sig_mask_arr,
    bands,
    comparison_label="Pink Noise Cont. vs. Music Cont.",
    mode='append',
    n_channels=len(networks),  # 3, not 128
    
)

print(latex_code)

print("Raw (uncorrected) group-level p-values, continuous music vs continuous pink noise:")
for band in raw_pvals:
    for net in raw_pvals[band]:
        flag = "  <-- p < 0.05 uncorrected" if raw_pvals[band][net] < 0.05 else ""
        print(f"  {band:10s} {net:25s} p = {raw_pvals[band][net]:.4f}{flag}")

\midrule

\multirow{5}{*}{\shortstack[l]{Pink Noise Cont. vs.\\Music Cont.}}
& $\delta$ & 0/3 & 0/3 & 0/3 ($\geq$2/3) & 0/3 \\
& $\theta$ & 0/3 & 0/3 & 0/3 ($\geq$2/3) & 0/3 \\
& $\alpha$ & 0/3 & 0/3 & 0/3 ($\geq$2/3) & 0/3 \\
& $\beta$ & 0/3 & 0/3 & 0/3 ($\geq$2/3) & 0/3 \\
& $\gamma$ & 0/3 & 1/3 & 0/3 ($\geq$2/3) & 0/3 \\


Raw (uncorrected) group-level p-values, continuous music vs continuous pink noise:
  delta      Sensorimotor Network      p = 0.0352  <-- p < 0.05 uncorrected
  delta      Default Mode Network      p = 0.8209
  delta      Frontoparietal Network    p = 0.6657
  theta      Sensorimotor Network      p = 0.4914
  theta      Default Mode Network      p = 0.4654
  theta      Frontoparietal Network    p = 0.3173
  alpha      Sensorimotor Network      p = 0.8734
  alpha      Default Mode Network      p = 0.7889
  alpha      Frontoparietal Network    p = 0.8099
  beta       Sensorimotor Network      p = 0.1384
  beta       Default Mode Network      p = 0.1767
  beta       

Aperiodic

In [44]:
eeg_fooof_pink_noise_reward_aperiodic = get_fooof_aperiodic(eeg_fooof_pink_noise_reward_matched)

fdr_corrected_p_vals_pink_noise_reward_aperiodic = get_band_power_difference_all_subjects_significance_aperiodic_fdr(eeg_fooof_pink_noise_reward_aperiodic, pat_state, n_channels=n_regions)



raw_pvals_ap, cluster_list_ap, cluster_means_ap = cluster_and_compare_fooof(eeg_fooof_pink_noise_reward_aperiodic , cond1_name="pink_noise",cond2_name="reward",
    mapping_dict=region_to_network, is_anatomical=False, avg_method='mean', alpha=0.05, abs_rel=2)




cluster_means_ap_off = {}

for sub in cluster_means_ap:
    cluster_means_ap_off[sub] = {}

    for parad in cluster_means_ap[sub]:
        cluster_means_ap_off[sub][parad] = {}
        cluster_means_ap_off[sub][parad]["Exponent"] = {}
        cluster_means_ap_off[sub][parad]["Offset"] = {}

        for net in cluster_means_ap[sub][parad]["Offset"]:
            cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 10)
            cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])

            # if sub == "sub03":
            #     cluster_means_ap_off[sub][parad]["Offset"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Offset"][net] + 20)
            #     cluster_means_ap_off[sub][parad]["Exponent"][net] = copy.deepcopy(cluster_means_ap[sub][parad]["Exponent"][net])



cluster_means_filtered = filter_networks(
    cluster_means_ap_off,
    networks_to_keep=NETWORKS
)


plots_path_power = plots_path + "fdr_source/fc_powers_music_cont_pink_noise_cont_all/ap/"

save_name = ""

plot_band_power_comparisons(cluster_means=cluster_means_filtered, condition1="pink_noise", condition2="reward", bands=['Offset', 'Exponent'],
    alpha_level=0.05, n_bootstrap=1000, figsize=(3, 5), legend_label1="Pink Noise Cont.", legend_label2="Music Cont.", save_dir=plots_path_power, 
    show_legend=False, power_unit=r"Aperiodic Parameters (a.u.)", save=True, save_tag=save_name, pat_state=pat_state_dict, FS_PVAL = 18, FS_YLAB = 20, FS_BAND = 20, FS_TITLE = 20, fdr_scope="subject")


stack_subject_figures_pdf(folder_path=plots_path_power, output_path=plots_path_power + "merged_all/src_aperiodic_violin_pinkcont_musiccont.pdf")



Aperiodic Components:  ['Offset', 'Exponent']
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
Paradigm 0:  pink_noise
Paradigm 1:  reward
✅ Done! Used Functional Network clustering
Number of clusters: 8
Running bootstrap tests …
Done.
Legend saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_music_cont_pink_noise_cont_all/ap/merged_all/legend_pink_noise_vs_reward_.pdf
Saved → D:/Taran/OneDrive - Radboud Universiteit/Documents/music_doc_study/analysis/plots_vector_nofilt/fdr_source/fc_powers_music_cont_pink_noise_cont_all/ap/merged_all/src_aperiodic_violin_pinkcont_musiccont.pdf


In [45]:
common = dict(cluster_means=cluster_means_filtered, condition1="pink_noise", condition2="reward",
              bands=['Offset', 'Exponent'],
              alpha_level=0.05, n_bootstrap=1000, save=False, pat_state=pat_state_dict)
p_net = plot_band_power_comparisons(**common, fdr_scope="network")
p_sub = plot_band_power_comparisons(**common, fdr_scope="subject")
changed = 0
for sub in p_net:
    for band in p_net[sub]:
        for net in p_net[sub][band]:
            if (p_net[sub][band][net] < 0.05) != (p_sub[sub][band][net] < 0.05):
                changed += 1
                print(f"{sub} {band:10s} {net:25s} network-FDR p={p_net[sub][band][net]:.3f} -> subject-FDR p={p_sub[sub][band][net]:.3f}")
print(f"{changed} significance decisions changed (continuous music vs pink noise, aperiodic)")

Running bootstrap tests …
Done.
Running bootstrap tests …
Done.
0 significance decisions changed (continuous music vs pink noise, aperiodic)
